In [1]:
import os, re, json, tqdm, csv, copy, math, gensim
import numpy as np
from nltk.probability import FreqDist
from scipy.special import kl_div, rel_entr
from gensim.models import Word2Vec, KeyedVectors
from zipfile import ZipFile
from reduce_pos import reduce_tag

# Feature extraction - COHA

### Frequencies of compound, constituents and paraphrase

In [2]:
path_to_corpus = "/mount/resources/corpora/COHA/CCOHA/tagged/"
def year_to_bin(year: int):
    if year < 1750:
        return 1
    elif year < 1800:
        return 2
    elif year < 1850:
        return 3
    elif year < 1900:
        return 4
    elif year < 1950:
        return 5
    else: return 6
MIN_YEAR = 1810
MAX_YEAR = 2010

In [3]:
timeslices = range(3, 7)
#my_compounds = ["glass tube", "fore part", "platinum wire", "copper wire", "iron wire", "salt solution", "blood pressure", "platinum black", "mineral manure", "induction coil", "mineral matter", "glass plate", "unit volume", "shoulder girdle", "absorption band", "cross section", "surface tension", "radius vector"]
my_compounds = [line.strip() for line in open("/home/users1/zabereus/zabereus/master_env/thesis/targets/compound_list_final.txt", "r")]
#topics = ['Agriculture', 'Anatomy 1', 'Anatomy 2', 'Astronomy', 'Atomic Physics', 'Biochemistry', 'Biography', 'Biology 1', 'Biology 2', 'Biology 3', 'Botany 1', 'Botany 2', 'Chemistry 1', 'Chemistry 2', 'Electricity', 'Fluid Dynamics', 'Formulae', 'Geography', 'Headmatter', 'Immunology', 'Measurement', 'Meteorology', 'Nervous System', 'Neurology', 'Optics', 'Paleontology', 'Physiology', 'Reporting', 'Tables', 'Thermodynamics']
topics = range(30)
lda_model = gensim.models.ldamodel.LdaModel.load("lda_coha/lda_coha.model")
id2word =  gensim.corpora.dictionary.Dictionary.load("lda_coha/lda_coha.model.id2word")

## Load existing features

In [11]:
with open("all_features_coha.json", "r") as f:
    all_features = json.load(f)

In [5]:
feature_scheme = {"freq_bin" + str(i): 0 for i in timeslices}
feature_scheme.update({"para_freq_bin" + str(i): 0 for i in timeslices})
feature_scheme.update({"mod_surprisals": {slice: [] for slice in timeslices}, "head_surprisals": {slice: [] for slice in timeslices}, "years": [], "topics": {topic:0 for topic in topics}})
#TODO maybe make years a dict too

all_features = {c: copy.deepcopy(feature_scheme) for c in my_compounds}
#example_sentences = {comp:[] for comp in my_compounds}
all_constituents = set([const for c in my_compounds for const in c.split()])

const_feature_scheme = {"freq_bin" + str(i): 0 for i in timeslices}
#const_feature_scheme.update({"prodset_as_head": set(), "prodset_as_mod": set()})
all_constituent_features = {const: copy.deepcopy(const_feature_scheme) for const in all_constituents}
tokens_per_year = {y:0 for y in range(MIN_YEAR, MAX_YEAR)}
tokens_per_topic = {topic:0 for topic in topics}

In [6]:
def get_features_from_sents(sentences, year, topic):
    """
    Takes in sentences (of a document) as list of lists of (token, pos) tuples
    and extracts features that get added to all_features and all_constituent_features
    """
    if not year:
        return
    bin = year_to_bin(year)
    global all_features, all_constituent_features
    for tokens in sentences:
        for i in range(len(tokens)):
            # count constituent occurrences (independent of whether they are part of a compound)
            if tokens[i][0] in all_constituent_features:
                all_constituent_features[tokens[i][0]]["freq_bin" + str(bin)] += 1
            if i == len(tokens)-1:
                break               # so we don't oob when checking the next token

            if tokens[i][1] == 'NN' and tokens[i+1][1] == 'NN' and (i==0 or tokens[i-1][1] != 'NN') and (i==len(tokens)-2 or tokens[i+2][1] != 'NN'):
                head = tokens[i+1][0].lower()
                modifier = tokens[i][0].lower()
                
                # For productivity
                # if head in all_constituent_features:
                #     all_constituent_features[head]["prodset_as_head"].add(modifier)
                # if modifier in all_constituent_features:
                #     all_constituent_features[modifier]["prodset_as_mod"].add(head)
                
                compound = modifier + " " + head
                if compound in my_compounds:
                    all_features[compound]["freq_bin" + str(bin)] += 1
                    all_features[compound]["years"].append(year)
                    try:
                        mod_surprisal = get_surprisal([token[0].lower().strip() for token in tokens[:i+1][-4:]])
                        head_surprisal = get_surprisal([token[0].lower().strip() for token in tokens[:i+2][-4:]])
                        # print(tokens[:i+1][-4:], mod_surprisal)
                        # print(tokens[:i+2][-4:], head_surprisal)
                    except KeyError:
                        print(tokens, i)
                        raise Exception
                    all_features[compound]["mod_surprisals"][bin].append(mod_surprisal)
                    all_features[compound]["head_surprisals"][bin].append(head_surprisal)
                    if topic is not None:
                        all_features[compound]["topics"][topic] += 1
                    # if len(tokens) > 8 and len(tokens) < 30:
                    #     example_sentences[compound].append(" ".join([token[0] for token in tokens]))
                    #print(f"compound {compound} found!")

            # paraphrases
            #check: line start/not a noun, then NN, then of/in, then NN, then line ends/not a noun
            if i < len(tokens)-2 and (i==0 or tokens[i-1][1] != 'NN') and tokens[i][1] == 'NN' \
                and (tokens[i+1][0] == 'of' or tokens[i+1][0] == 'in') and tokens[i+2][1] == 'NN' \
                and (i == len(tokens)-3 or tokens[i+3][1] != 'NN'): 
                compound = tokens[i+2][0].lower() + " " + tokens[i][0].lower()
                bin = year_to_bin(year)
                if compound in all_features:
                    all_features[compound]["para_freq_bin" + str(bin)] += 1
                    #print(f"para {compound} found!")

            # paraphrases with determiners
            #check: line start/not a noun, then NN, then of/in, then DET, then NN, then line ends/not a noun
            if i < len(tokens)-3 and (i==0 or tokens[i-1][1] != 'NN') and tokens[i][1] == 'NN' \
                and (tokens[i+1][0] == 'of' or tokens[i+1][0] == 'in') and tokens[i+2][1] == 'DT' and tokens[i+3][1] == 'NN' \
                and (i == len(tokens)-4 or tokens[i+4][1] != 'NN'): 
                compound = tokens[i+3][0].lower() + " " + tokens[i][0].lower()
                bin = year_to_bin(year)
                if compound in all_features:
                    all_features[compound]["para_freq_bin" + str(bin)] += 1
                    #print(f"paradet {compound} found!")

In [6]:
# for testing purposes
def get_texts_per_bin(decades=range(1810, 2010, 10)):
    texts_per_bin = {bin: 0 for bin in timeslices}
    for decade in decades:
        with ZipFile(path_to_corpus + f'cleaned_{decade}s.zip') as zf:
            texts_per_bin[year_to_bin(decade)] += len(zf.namelist())
    print(texts_per_bin)  
    print(sum(texts_per_bin.values()))
get_texts_per_bin()

{3: 1621, 4: 9259, 5: 39588, 6: 66146}
116614


In [ ]:
def get_features_from_coha(decades=range(1810, 2010, 10)):
    for decade in decades:
        with ZipFile(path_to_corpus + f'cleaned_{decade}s.zip') as zf:
            for file in zf.namelist():
                year = int(file.split('_')[1])
                with zf.open(file) as f:
                    sentences = []
                    tokens = []
                    just_tokens = []
                    for line in f.readlines():
                        token, _, pos = line.decode().strip().split('\t')
                        if token == "<eos>":
                            #OLD continue
                            sentences.append(tokens)
                            tokens = []
                        elif token != '@':
                            just_tokens.append(token)
                            tokens.append((token, reduce_tag(pos)))
                    #bow = id2word.doc2bow(gensim.utils.simple_preprocess(' '.join(just_tokens), deacc=True))
                    #primary_topic = np.argmax([x[1] for x in lda_model[bow]])
                    get_features_from_sents(sentences, year, 0) #TODO don't forget to put primary topic back
                    #tokens_per_year[year] += len(just_tokens)
                    #tokens_per_topic[primary_topic] += len(just_tokens)
        print(f"Decade {decade} done")

get_features_from_coha()

In [36]:
print(old_features["salt water"])

{'freq_bin3': 1.938188421708603, 'freq_bin4': 1.5677851870831572, 'freq_bin5': 1.9124645766455108, 'freq_bin6': 1.7692613753750697, 'para_freq_bin3': 0.0, 'para_freq_bin4': 0.0, 'para_freq_bin5': 0.0, 'para_freq_bin6': 0.0, 'years': [1812, 1815, 1827, 1821, 1821, 1827, 1822, 1823, 1823, 1823, 1823, 1821, 1823, 1823, 1839, 1835, 1835, 1835, 1835, 1831, 1831, 1831, 1831, 1831, 1831, 1832, 1835, 1838, 1839, 1835, 1838, 1838, 1839, 1836, 1835, 1837, 1833, 1838, 1835, 1835, 1835, 1839, 1839, 1839, 1839, 1836, 1836, 1836, 1839, 1832, 1835, 1835, 1834, 1838, 1839, 1847, 1847, 1841, 1848, 1848, 1841, 1843, 1844, 1849, 1843, 1841, 1841, 1848, 1848, 1848, 1842, 1843, 1847, 1847, 1847, 1845, 1841, 1849, 1849, 1849, 1849, 1849, 1849, 1849, 1849, 1849, 1854, 1859, 1857, 1851, 1859, 1856, 1856, 1855, 1850, 1855, 1853, 1853, 1855, 1854, 1854, 1856, 1851, 1854, 1850, 1850, 1855, 1867, 1863, 1863, 1863, 1863, 1863, 1863, 1860, 1867, 1867, 1867, 1868, 1868, 1869, 1866, 1869, 1869, 1869, 1860, 1869, 1863

In [ ]:
#print(all_features)
no_occs = []
all_occs = []
for compound, features in all_features.items():
    occs = sum([occ for feat, occ in features.items() if feat.startswith('freq')])
    para_occs = sum([occ for feat, occ in features.items() if feat.startswith('para')])
    all_occs.append((occs, para_occs))
    if occs + para_occs > 0:
        print(compound, f": Occurrs {occs} times, paraphrase occurs {para_occs} times")
    if occs == 0: no_occs.append(compound)
print(len(no_occs), " compounds do not appear in this sample")
#print(compound_occurrences)
for compound in no_occs:
    del all_features[compound]
print(f"{len(no_occs)} compounds removed from feature set, {len(all_features)} compounds remaining")

In [28]:
print(all_features["salt water"])
print(old_features["salt water"])

{'freq_bin3': 86, 'freq_bin4': 169, 'freq_bin5': 265, 'freq_bin6': 320, 'para_freq_bin3': 0, 'para_freq_bin4': 0, 'para_freq_bin5': 0, 'para_freq_bin6': 0, 'surprisals': {3: [32.464675050040995, 44.09804728536671, 25.904591249010203, 27.304576347952587, 36.44278493521699, 30.94219572496841, 44.64609512209994, 40.66190079607439, 38.413631198381694, 42.85785458189442, 34.24906975818232, 29.062556986346607, 45.87738593031252, 36.579681037648974, 29.92771853661117, 33.660396988105575, 35.84269001312466, 34.07056445587806, 30.113842796654563, 40.11970500633348, 41.66415770789123, 41.633732686860704, 32.140401299854645, 41.60413969816497, 41.65531850927268, 34.14819516416728, 38.32289466367794, 20.289729353676073, 35.528650487372815, 38.80194097854345, 36.86261923541205, 34.98730941001846, 30.18248458453579, 30.141441768721002, 38.43598242325863, 36.69180993867992, 39.18584670391032, 32.84341513375981, 41.89871196771805, 36.272437045144734, 30.33346796029781, 34.99893998589474, 28.1139984387

In [27]:
for compound, features in old_features.items():
    features.update({key: value for key, value in all_features[compound].items() if key.startswith("cosine")})

### Secondary Processing

In [25]:
def kld_norm(occurrences, target_dist, topics = False):
    if topics:
        dist = [occs / sum(occurrences) for occs in occurrences] # normalize distribution
    else: # years
        dist = [0 for _ in range(MIN_YEAR, MAX_YEAR)]
        for year in occurrences:
            dist[year-MIN_YEAR] += 1/len(occurrences) # normalize while adding

    #TODO we could also replace this implementation with a base 2 log one
    kld = sum(rel_entr(dist, target_dist))
    return 1 - math.exp(-kld)

In [26]:
def get_tokens_per_bin():
    tokens_per_bin = {bin: 0 for bin in timeslices}
    for year, tokens in tokens_per_year.items():
        tokens_per_bin[year_to_bin(year)] += tokens
    return {bin: val/1e6 for bin, val in tokens_per_bin.items()}
print(get_tokens_per_bin())

{3: 44.371331, 4: 107.79538, 5: 138.564658, 6: 181.431644}


In [27]:
# Global distributions
tokens_per_year_dist = [occs / sum(tokens_per_year.values()) for occs in tokens_per_year.values()] # normalize distribution
tokens_per_topic_dist = [occs / sum(tokens_per_year.values()) for occs in tokens_per_topic.values()] # normalize distribution

tokens_per_bin = get_tokens_per_bin()

# year and topic distribution per compound
for compound, features in all_features.items():
    # constituent frequencies
    head, modifier = compound.split()
    for bin in timeslices:
        # add constituent freqs, convert all freqs to relative frequencies
        features["head_freq_bin" + str(bin)] = all_constituent_features[head]["freq_bin" + str(bin)] / tokens_per_bin[bin]
        features["mod_freq_bin" + str(bin)] = all_constituent_features[modifier]["freq_bin" + str(bin)] / tokens_per_bin[bin]
        features["freq_bin" + str(bin)] /= tokens_per_bin[bin] 
        features["para_freq_bin" + str(bin)] /= tokens_per_bin[bin]

    # calculate divergences
    features["temp_divergence"] = kld_norm(features["years"], tokens_per_year_dist)
    features["topic_divergence"] = kld_norm(features["topics"].values(), tokens_per_topic_dist, topics=True)
    
    # average surprisal values
    for bin, surps in features["mod_surprisals"].items():
        features["avg_mod_surprisal_bin" + str(bin)] = 0 if len(surps) == 0 else sum(surps)/len(surps)
    for bin, surps in features["head_surprisals"].items():
        features["avg_head_surprisal_bin" + str(bin)] = 0 if len(surps) == 0 else sum(surps)/len(surps)
    #del features["years"], features["topics"], features["surprisals"]
    #print(features["temp_divergence"], features["topic_divergence"], features["avg_surprisal"])

In [32]:
print(all_features['cut surface'])
print(all_constituent_features['glass'])

{'freq_bin3': 0.022537074671030267, 'freq_bin4': 0.009276835426527557, 'freq_bin5': 0.02165054237711899, 'freq_bin6': 0.027558588401480833, 'para_freq_bin3': 0.0, 'para_freq_bin4': 0.0, 'para_freq_bin5': 0.0, 'para_freq_bin6': 0.0, 'surprisals': {'3': [0.0], '4': [0.0], '5': [0.0, 0.0, 0.0], '6': [0.0, 0.0, 0.0, 0.0, 0.0]}, 'years': [1845, 1880, 1906, 1946, 1946, 1975, 1977, 1988, 1990, 2004], 'topics': {'0': 3, '1': 0, '2': 1, '3': 2, '4': 0, '5': 2, '6': 1, '7': 0, '8': 1, '9': 0, '10': 0, '11': 0, '12': 0, '13': 0, '14': 0, '15': 0, '16': 0, '17': 0, '18': 0, '19': 0, '20': 0, '21': 0, '22': 0, '23': 0, '24': 0, '25': 0, '26': 0, '27': 0, '28': 0, '29': 0}, 'head_freq_bin3': 102.9042829479242, 'mod_freq_bin3': 64.97438627658026, 'head_freq_bin4': 108.91004790743351, 'mod_freq_bin4': 61.51469571330423, 'head_freq_bin5': 143.4348432484133, 'mod_freq_bin5': 64.59800160586403, 'head_freq_bin6': 159.98311738827655, 'mod_freq_bin6': 66.36108087076585, 'temp_divergence': 0.9488191434337812

### Context vectors

In [5]:
def replace_compounds(tokens: list):
    """
    Takes in tokens as a list of (token, pos, *) and returns a list of tokens as strings, with compouds in my_compounds
    as 1 token connected by an underscore
    """
    tokens_new = []
    skip_next = False
    for i in range(len(tokens)):
        if skip_next:
            skip_next = False
            continue
        if i == len(tokens)-1:
            tokens_new.append(tokens[i][0])
        elif tokens[i][1] == 'NN' and tokens[i+1][1] == 'NN' and (i==0 or tokens[i-1][1] != 'NN') and (i==len(tokens)-2 or tokens[i+2][1] != 'NN'):
            # compound detected
            head = tokens[i+1][0].lower()
            modifier = tokens[i][0].lower()
            compound = modifier + " " + head
            if compound in my_compounds:
                tokens_new.append(f"{modifier}_{head}") # replace constituents with compound
                skip_next = True # makes sure head doesn't get added additionally
            else: 
                tokens_new.append(tokens[i][0])
        else:  # not compound  
                tokens_new.append(tokens[i][0])
    return tokens_new

def sentence_reader_coha(bin):
    """
    Generator yielding COHA documents as strings, tokens separated by spaces
    """
    path_to_corpus = "/mount/resources/corpora/COHA/CCOHA/tagged/"
    decades = [dec for dec in range(1810, 2010, 10) if year_to_bin(dec) == bin]
    for decade in decades:
        with ZipFile(path_to_corpus + f'cleaned_{decade}s.zip') as zf:
            for file in zf.namelist():
                with zf.open(file) as f:
                    tokens = []
                    for line in f.readlines():
                        token, _, pos = line.decode().strip().split('\t')
                        if token == "<eos>":
                            #print(replace_compounds(tokens))
                            sent = replace_compounds(tokens)
                            if len(sent) > 2:
                                yield sent
                            tokens = []
                        else:
                            tokens.append((token, reduce_tag(pos), 0))
        print(f"decade {decade} done")

In [6]:
vectors = Word2Vec(list(sentence_reader_coha(5)), min_count=1, workers=10).wv
vectors.save(f"w2v3_coha/word2vec_bin5.wordvectors")

decade 1950 done
decade 1960 done
decade 1970 done
decade 1980 done
decade 1990 done
decade 2000 done


In [6]:
for bin in tqdm.tqdm([3, 4, 5, 6]):
    vectors = Word2Vec(list(sentence_reader_coha(bin)), min_count=1, workers=10).wv
    vectors.save(f"w2v3_coha/word2vec_bin{bin}.wordvectors")

  0%|          | 0/4 [00:00<?, ?it/s]

decade 1810 done
decade 1820 done
decade 1830 done
decade 1840 done


 25%|██▌       | 1/4 [04:12<12:36, 252.31s/it]

decade 1850 done
decade 1860 done
decade 1870 done
decade 1880 done
decade 1890 done


 50%|█████     | 2/4 [14:47<15:55, 477.68s/it]

decade 1900 done
decade 1910 done
decade 1920 done
decade 1930 done
decade 1940 done


 75%|███████▌  | 3/4 [29:25<11:00, 660.26s/it]

decade 1950 done
decade 1960 done
decade 1970 done
decade 1980 done
decade 1990 done
decade 2000 done


100%|██████████| 4/4 [49:49<00:00, 747.37s/it]


In [23]:
# Get average cosine similarities for use as default values
all_mod_cosines = {bin: [] for bin in timeslices}
all_head_cosines = {bin: [] for bin in timeslices}

for bin in timeslices:
    vectors = KeyedVectors.load(f"w2v3_coha/word2vec_bin{bin}.wordvectors")
    for compound in all_features.keys():
        mod, head = compound.split()
        if mod + "_" + head in vectors and mod in vectors:
            all_mod_cosines[bin].append(float(vectors.similarity(mod + "_" + head, mod)))
        if mod + "_" + head in vectors and head in vectors:
            all_head_cosines[bin].append(float(vectors.similarity(mod + "_" + head, head)))

mod_cos_avgs = [sum(sims)/len(sims) for sims in all_mod_cosines.values()]
head_cos_avgs = [sum(sims)/len(sims) for sims in all_head_cosines.values()]
print(mod_cos_avgs, head_cos_avgs)

[0.23536423938814552, 0.1840367341011932, 0.18360054879162613, 0.20851945765488952] [0.23776238322257995, 0.18808840298449717, 0.20205798875731626, 0.20705437680223474]


In [25]:
mod_cos_avgs = [0, 0, 0.23932273814454674, 0.19943520368674814, 0.19113830771338916, 0.21876911534151683] 
head_cos_avgs = [0, 0, 0.21635047912597657, 0.19749461537735027, 0.21299252577070424, 0.21060386983401796]
mod_cos_avgs2 = [0, 0, 0.24671881446614863, 0.20001029441217807, 0.18650108863861967, 0.2129492657224613]
head_cos_avgs2 = [0, 0, 0.24043643645942211, 0.181263623440662, 0.21231736138431745, 0.20814135126528088]
mod_cos_avgs3 = [0, 0, 0.23536423938814552, 0.1840367341011932, 0.18360054879162613, 0.20851945765488952]
head_cos_avgs_3 = [0, 0, 0.23776238322257995, 0.18808840298449717, 0.20205798875731626, 0.20705437680223474]

for vec in ["", "2", "3"]:
    mod_avg, head_avg = {"": (mod_cos_avgs, head_cos_avgs), "2": (mod_cos_avgs2, head_cos_avgs2), "3": (mod_cos_avgs3, head_cos_avgs_3) }[vec]
    for bin in timeslices:
        vectors = KeyedVectors.load(f"w2v{vec}_coha/word2vec_bin{bin}.wordvectors")
        for compound in all_features.keys():
            mod, head = compound.split()
            if mod + "_" + head in vectors:
                all_features[compound][f"cosine{vec}_mod_bin{bin}"] = float(vectors.similarity(mod + "_" + head, mod)) if mod in vectors else mod_avg[bin-1]
                all_features[compound][f"cosine{vec}_head_bin{bin}"] = float(vectors.similarity(mod + "_" + head, head)) if head in vectors else head_avg[bin-1]
            else:
                all_features[compound][f"cosine{vec}_mod_bin{bin}"] = mod_avg[bin-1]
                all_features[compound][f"cosine{vec}_head_bin{bin}"] = head_avg[bin-1]

In [ ]:
#for ix, model in enumerate(bin_models):
#    model.save(f"word2vec_bin{ix+1}.model")

## N-gram surprisal model

In [5]:
def get_all_contexts(decades = range(1810, 2010, 10)):
    """
    idea: Iterate through corpus once, grabbing all contexts that occur before our compounds
    Then iterate a second time, grabbing all instances of those contexts
    """
    occuring_trigram_contexts = []
    for decade in decades:
        with ZipFile(path_to_corpus + f'cleaned_{decade}s.zip') as zf:
            for file in zf.namelist():
                with zf.open(file) as f:
                    tokens = []
                    for line in f.readlines():
                        token = line.decode().strip().split('\t')[0]
                        if token == "<eos>":
                            trigram_contexts = get_compound_contexts_from_sent(tokens)
                            occuring_trigram_contexts.extend(trigram_contexts)
                            tokens = []
                        elif token != '@':
                            tokens.append(token.lower())
        print(f"Decade {decade} done")
    occuring_trigram_contexts = set(occuring_trigram_contexts)
    print(f"{len(occuring_trigram_contexts)} trigram contexts occur.")
    return occuring_trigram_contexts


def get_compound_contexts_from_sent(tokens):
    """
    Takes in a sentence as list of tokens and retuns all contexts that occur before a target compound
    """
    trigram_contexts = []
    tokens = 3 * ["<null>"] + tokens + ["<null>"]

    sentence_5grams = [(tokens[i], tokens[i+1], tokens[i+2], tokens[i+3], tokens[i+4]) for i in range(len(tokens) - 4)]

    for tetragram in sentence_5grams:
        if tetragram[3] in all_constituent_features: #for speed
            if tetragram[2] + " " + tetragram[3] in my_compounds or tetragram[3] + " " + tetragram[4] in my_compounds:
                trigram_contexts.append((tetragram[0], tetragram[1], tetragram[2]))

    return trigram_contexts
                    
def get_ngram_probs(tris, decades = range(1810, 2010, 10)):
    trigram_probs ={trigram: {"other": 0} for trigram in tris}
    for decade in decades:
        with ZipFile(path_to_corpus + f'cleaned_{decade}s.zip') as zf:
            for file in zf.namelist():
                with zf.open(file) as f:
                    tokens = []
                    for line in f.readlines():
                        token = line.decode().strip().split('\t')[0]
                        if token != '@':
                            tokens.append(token.lower())
                        if token == "<eos>":
                            get_contexts_from_sent(tokens, trigram_probs)
                            tokens = []        
        print(f"Decade {decade} done")
    return normalize_dist(trigram_probs)

def get_contexts_from_sent(tokens, trigram_probs):
    """
    Takes in a sentence as list of tokens (incl <eos> at the end) and updates context probabilities
    """
    tokens = 3 * ["<null>"] + tokens
    sentence_tetragrams = [(tokens[i], tokens[i+1], tokens[i+2], tokens[i+3]) for i in range(len(tokens) - 3)]
    trigrams = trigram_probs.keys()

    for tetragram in sentence_tetragrams:
        if (tetragram[0], tetragram[1], tetragram[2]) in trigrams:
            if tetragram[3] in all_constituents:
                if tetragram[3] in trigram_probs[(tetragram[0], tetragram[1], tetragram[2])]:
                    trigram_probs[(tetragram[0], tetragram[1], tetragram[2])][tetragram[3]]
                else:
                    trigram_probs[(tetragram[0], tetragram[1], tetragram[2])][tetragram[3]] = 1
            else:
                trigram_probs[(tetragram[0], tetragram[1], tetragram[2])]["other"] += 1    

            
def normalize_dist(dist: dict):
    return {context: {token: prob/sum(probdist.values()) if sum(probdist.values()) else 0 for token, prob in probdist.items()} for context, probdist in dist.items()}


In [6]:
trigram_contexts = get_all_contexts()

Decade 1810 done
Decade 1820 done
Decade 1830 done
Decade 1840 done
Decade 1850 done
Decade 1860 done
Decade 1870 done
Decade 1880 done
Decade 1890 done
Decade 1900 done
Decade 1910 done
Decade 1920 done
Decade 1930 done
Decade 1940 done
Decade 1950 done
Decade 1960 done
Decade 1970 done
Decade 1980 done
Decade 1990 done
Decade 2000 done
16174 trigram contexts occur.


In [7]:
trigram_dist = get_ngram_probs(trigram_contexts)
#json_bigram_dist = {" ".join(key): value for key, value in bigram_dist.items()}
json_trigram_dist = {" ".join(key): value for key, value in trigram_dist.items()}
with open("ngram_dists.json", "w") as f:
    json.dump(json_trigram_dist, f)

Decade 1810 done
Decade 1820 done
Decade 1830 done
Decade 1840 done
Decade 1850 done
Decade 1860 done
Decade 1870 done
Decade 1880 done
Decade 1890 done
Decade 1900 done
Decade 1910 done
Decade 1920 done
Decade 1930 done
Decade 1940 done
Decade 1950 done
Decade 1960 done
Decade 1970 done
Decade 1980 done
Decade 1990 done
Decade 2000 done


In [7]:
with open("ngram_dists.json", "r") as f:
    json_trigram_dist = json.load(f)
trigram_dist = {tuple(key.split()): value for key, value in json_trigram_dist.items()}
del json_trigram_dist

In [8]:
def get_surprisal(tokens: list):
    """
    Inverse log probability of last token given 3 tokens context
    """
    all_tetragram_probs = trigram_dist

    #print("surprisal of tokens: ", tokens)
    tokens = 3 * ["<null>"] + tokens
    prob = all_tetragram_probs[(tokens[-4], tokens[-3], tokens[-2])][tokens[-1]]

    return - math.log(prob) ##word_prob * bigram_prob * trigram_prob * tetragram_prob)

In [24]:
print(trigram_dist["a", "material"])

KeyError: ('a', 'material')

## Neighborhood density

In [13]:
averages = [0, 0, 0.7099209564328194, 0.7051741580394181, 0.7045321619332727, 0.7022177097090969]

for bin in timeslices:
    vectors = KeyedVectors.load(f"w2v2_coha/word2vec_bin{bin}.wordvectors")
    for compound in all_features.keys():
        compound_ = "_".join(compound.split())
        if compound_ in vectors:
            most_similar = vectors.most_similar(positive=compound_, topn=10)
            avg = np.average([token[1] for token in most_similar])
        else:
            avg = averages[bin-1]
        all_features[compound][f"neighborhood_density_bin{bin}"] = avg  #high similarity means high density
print(all_features["salt water"]["neighborhood_density_bin3"])
print(all_features["salt water"]["neighborhood_density_bin6"])
print(all_features["blood pressure"]["neighborhood_density_bin3"])
print(all_features["blood pressure"]["neighborhood_density_bin6"])

0.7357436299324036
0.7863499224185944
0.7099209564328194
0.7307724118232727


In [8]:
# get averages
neighborhood_sims = []
for bin in timeslices:
    sims_slices = []
    vectors = KeyedVectors.load(f"w2v2_coha/word2vec_bin{bin}.wordvectors")
    for compound in all_features.keys():
        compound_ = "_".join(compound.split())
        if compound_ in vectors:
            most_similar = vectors.most_similar(positive=compound_, topn=10)
            avg = np.average([token[1] for token in most_similar])
            sims_slices.append(avg)
    print(np.average(sims_slices))
    neighborhood_sims.append(np.average(sims_slices))


0.7099209564328194
0.7051741580394181
0.7045321619332727
0.7022177097090969


## Saving all features

In [14]:
with open("all_features_coha.json", "w") as f:
    json.dump(all_features, f)

#with open("distributions_coha.json", "w") as exs_f:
#    json.dump([tokens_per_year_dist, tokens_per_topic_dist], exs_f)